# Lee Language Lab Candidate Challenge — Part 1
## Partial Freezing of a Multilingual Distilled Model for Persian POS Tagging

**Purpose.** Compare full fine-tuning against a controlled partial-freezing condition on the Persian UD PerDT treebank.

**Primary comparison:**
- Baseline: all DistilmBERT encoder layers + classification head are trainable.
- Partial freezing: lower 3 of the 6 Transformer blocks are frozen; embeddings, upper 3 blocks, and token-classification head remain trainable.

**Important:** run every cell in order. Do not report results until the final Test evaluation has run. The Test split is kept untouched during model selection; validation accuracy selects the best epoch.

### Research integrity
- We will only report metrics produced by this notebook on your machine.
- Keep the generated `run_metadata.json`, training logs, class-level scores, and this notebook together.
- Record any changes to the dataset size, batch size, epochs, or model revision in the report.
- This is a first controlled experiment, not evidence by itself that freezing is universally better.

## 1. Experimental protocol

| Component | Fixed choice |
|---|---|
| Language / treebank | Persian — UD Persian PerDT |
| Task | Word-level UPOS tagging |
| Encoder | `distilbert/distilbert-base-multilingual-cased` |
| Train subset | Deterministic random prefix; starts at 5,000 sentences and expands only if a UPOS class is missing / underrepresented |
| Validation / Test | Entire official Dev and Test splits |
| Labels | 16 UPOS labels used by PerDT (the UD inventory has 17 possible labels; PerDT does not use `SYM`) |
| Max sequence length | 256 subword tokens (truncation is measured and reported) |
| Batch size | 8 |
| Epochs | 3 |
| Optimizer | AdamW, learning rate 2e-5, weight decay 0.01 |
| Seed | 42 |
| Model selection | Best validation token accuracy; final test is run once per condition |

To preserve a fair comparison, the two models are initialized with the same random seed, use identical examples and data order, and differ only in which lower Transformer blocks have `requires_grad=False`.

**Interpretation caveat:** the frozen condition still trains the embedding layer and the task head. It is therefore “freeze lower Transformer blocks,” not “freeze the whole lower half of the network including embeddings.”

## 2. Install and import dependencies

For a local Windows environment, install the packages from the accompanying `requirements.txt`. A GPU is useful but not mandatory. First run this notebook on CPU only if no GPU is available; expect training to take longer.

In [1]:
import os, sys, json, time, random, hashlib, platform, urllib.request
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
import transformers
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

Python: 3.11.9
PyTorch: 2.14.1+cu130
Transformers: 5.23.0
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
CUDA version: 13.0


## 3. Paths and fixed configuration

The UPOS list below is based on the tags documented for UD Persian PerDT. The treebank uses 16 of the 17 universal UPOS tags; `SYM` is not used.

In [2]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "raw"
RESULTS_DIR = PROJECT_DIR / "results"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TREEBANK_BASE_URL = (
    "https://raw.githubusercontent.com/UniversalDependencies/"
    "UD_Persian-PerDT/master"
)
DATA_FILES = {
    "train": "fa_perdt-ud-train.conllu",
    "validation": "fa_perdt-ud-dev.conllu",
    "test": "fa_perdt-ud-test.conllu",
}

MODEL_NAME = "distilbert/distilbert-base-multilingual-cased"
UPOS_TAGS = [
    "ADJ", "ADP", "ADV", "AUX", "CCONJ", "DET", "INTJ", "NOUN",
    "NUM", "PART", "PRON", "PROPN", "PUNCT", "SCONJ", "VERB", "X"
]
LABEL2ID = {label: i for i, label in enumerate(UPOS_TAGS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

SEED = 42
INITIAL_TRAIN_SENTENCES = 5_000
MAX_TRAIN_SENTENCES = 12_000
MIN_TRAIN_INSTANCES_PER_LABEL = 5
MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_FROZEN_LAYERS = 3
WARMUP_RATIO = 0.10

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Project directory:", PROJECT_DIR.resolve())
print("Device:", DEVICE)

Project directory: .
Device: cuda


## 4. Reproducibility helpers and run metadata

In [3]:
def set_global_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Deterministic settings improve repeatability, though some GPU kernels can still vary slightly.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


set_global_seed(SEED)
run_metadata = {
    "python": sys.version,
    "platform": platform.platform(),
    "pytorch": torch.__version__,
    "transformers": transformers.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "device": str(DEVICE),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cuda_version": torch.version.cuda,
    "seed": SEED,
    "model_name": MODEL_NAME,
    "model_revision": None,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "num_frozen_layers": NUM_FROZEN_LAYERS,
}

## 5. Download the official UD files

The official repository uses the standard CoNLL-U files. The script stores them locally and computes SHA-256 checksums so the exact downloaded files can be identified later. If downloading fails, use the three files from the official repository and place them in `data/raw/` with the exact names below.

In [4]:
def download_if_missing(filename: str) -> Path:
    destination = DATA_DIR / filename
    if destination.exists() and destination.stat().st_size > 0:
        print(f"Already present: {destination.name}")
        return destination

    url = f"{TREEBANK_BASE_URL}/{filename}"
    print("Downloading:", url)
    request = urllib.request.Request(url, headers={"User-Agent": "LeeLab-POS-Research-Project/1.0"})
    try:
        with urllib.request.urlopen(request, timeout=90) as response, destination.open("wb") as out:
            out.write(response.read())
    except Exception:
        if destination.exists():
            destination.unlink()
        raise
    if destination.stat().st_size == 0:
        raise RuntimeError(f"Downloaded file is empty: {filename}")
    return destination


file_paths = {split: download_if_missing(name) for split, name in DATA_FILES.items()}
file_hashes = {split: sha256_file(path) for split, path in file_paths.items()}
run_metadata["dataset_source"] = TREEBANK_BASE_URL
run_metadata["dataset_file_sha256"] = file_hashes
print(json.dumps(file_hashes, indent=2))

Downloading: https://raw.githubusercontent.com/UniversalDependencies/UD_Persian-PerDT/master/fa_perdt-ud-train.conllu
Downloading: https://raw.githubusercontent.com/UniversalDependencies/UD_Persian-PerDT/master/fa_perdt-ud-dev.conllu
Downloading: https://raw.githubusercontent.com/UniversalDependencies/UD_Persian-PerDT/master/fa_perdt-ud-test.conllu
{
  "train": "5399c9543e31a9fbe9d994292b7b73b51a6e304e73e308eadc3a5b02c8f29a6d",
  "validation": "6dd8614902dbe1e02e5fc3caae912982f3675378b14b5a1a71e6b2784917917b",
  "test": "4daf1fbf1bf9e0e36ac15c8ce1fe050740c7dc016cd0fdeb67b785699c1d599d"
}


## 6. Parse CoNLL-U and audit the data

CoNLL-U includes multiword-token rows (IDs such as `2-3`) and empty-node rows (IDs such as `5.1`). These are not independent syntactic words and are excluded. For each remaining integer-ID row, `FORM` is the input word and `UPOS` is the target label.

In [5]:
def parse_conllu(path: Path):
    sentences = []
    counters = Counter()
    current_words, current_tags = [], []

    def finish_sentence():
        nonlocal current_words, current_tags
        if current_words:
            sentences.append({"tokens": current_words, "tags": current_tags})
        current_words, current_tags = [], []

    with path.open("r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.rstrip("\r\n")
            if not line:
                finish_sentence()
                continue
            if line.startswith("#"):
                counters["comment_lines"] += 1
                continue

            fields = line.split("\t")
            if len(fields) != 10:
                counters["malformed_rows"] += 1
                continue

            token_id, form, upos = fields[0], fields[1], fields[3]
            if "-" in token_id:
                counters["multiword_token_rows"] += 1
                continue
            if "." in token_id:
                counters["empty_node_rows"] += 1
                continue
            try:
                int(token_id)
            except ValueError:
                counters["invalid_id_rows"] += 1
                continue

            if form == "_" or upos == "_":
                counters["missing_form_or_upos"] += 1
                continue
            if upos not in LABEL2ID:
                counters[f"unknown_upos_{upos}"] += 1
                continue

            current_words.append(form)
            current_tags.append(upos)
            counters["syntactic_word_rows"] += 1

    finish_sentence()
    counters["sentences"] = len(sentences)
    return sentences, counters


raw_splits, parser_audits = {}, {}
for split, path in file_paths.items():
    raw_splits[split], parser_audits[split] = parse_conllu(path)
    print(f"{split:10s}: {len(raw_splits[split]):,} sentences | {sum(len(x['tokens']) for x in raw_splits[split]):,} words")

print("\nParser audit:")
for split, audit in parser_audits.items():
    print(split, dict(audit))

# Basic integrity checks against the published treebank split sizes.
expected_sentence_counts = {"train": 26196, "validation": 1456, "test": 1455}
for split, expected in expected_sentence_counts.items():
    actual = len(raw_splits[split])
    if actual != expected:
        print(f"WARNING: {split} contains {actual} parsed sentences; published reference is {expected}. Investigate before training.")

for split, examples in raw_splits.items():
    assert examples, f"No examples parsed from {split}."
    observed = {tag for ex in examples for tag in ex["tags"]}
    unknown = observed - set(UPOS_TAGS)
    assert not unknown, f"Unexpected UPOS tags in {split}: {unknown}"

train_tag_counts = Counter(tag for ex in raw_splits["train"] for tag in ex["tags"])
print("\nFull training UPOS token counts:")
display(pd.DataFrame({"UPOS": UPOS_TAGS, "count": [train_tag_counts[t] for t in UPOS_TAGS]}))

train     : 26,196 sentences | 452,496 words
validation: 1,456 sentences | 25,147 words
test      : 1,455 sentences | 24,133 words

Parser audit:
train {'comment_lines': 78588, 'syntactic_word_rows': 452496, 'multiword_token_rows': 6508, 'sentences': 26196}
validation {'comment_lines': 4368, 'syntactic_word_rows': 25147, 'multiword_token_rows': 369, 'sentences': 1456}
test {'comment_lines': 4365, 'syntactic_word_rows': 24133, 'multiword_token_rows': 296, 'sentences': 1455}

Full training UPOS token counts:


,UPOS,count
0,ADJ,31811
1,ADP,64113
2,ADV,7460
3,AUX,16684
4,CCONJ,20149
5,DET,9409
6,INTJ,629
7,NOUN,151678
8,NUM,5594
9,PART,404


## 7. Select a manageable, deterministic training subset

We keep all official Dev and Test examples. The training subset is selected only from Train using a fixed random seed. It begins at 5,000 sentences and expands in 500-sentence increments if any of the 16 UPOS classes has fewer than five token instances. This avoids silently training with a missing class while keeping the dataset small enough for a short experiment.

In [6]:
def choose_training_subset(examples, initial_size=INITIAL_TRAIN_SENTENCES,
                           max_size=MAX_TRAIN_SENTENCES,
                           min_per_label=MIN_TRAIN_INSTANCES_PER_LABEL,
                           seed=SEED):
    rng = random.Random(seed)
    indices = list(range(len(examples)))
    rng.shuffle(indices)

    size = min(initial_size, len(examples))
    step = 500
    while True:
        selected = [examples[i] for i in indices[:size]]
        counts = Counter(tag for ex in selected for tag in ex["tags"])
        missing_or_rare = [tag for tag in UPOS_TAGS if counts[tag] < min_per_label]
        if not missing_or_rare or size >= min(max_size, len(examples)):
            return selected, size, counts, missing_or_rare
        size = min(size + step, max_size, len(examples))


train_examples, chosen_train_size, subset_tag_counts, rare_tags = choose_training_subset(raw_splits["train"])
validation_examples = raw_splits["validation"]
test_examples = raw_splits["test"]

print(f"Selected training sentences: {chosen_train_size:,} / {len(raw_splits['train']):,}")
print(f"Validation sentences: {len(validation_examples):,} (all official Dev)")
print(f"Test sentences: {len(test_examples):,} (all official Test)")
if rare_tags:
    print("WARNING — labels still underrepresented in selected Train subset:", rare_tags)
else:
    print("All 16 UPOS labels meet the minimum training count.")

display(pd.DataFrame({"UPOS": UPOS_TAGS, "train_subset_tokens": [subset_tag_counts[t] for t in UPOS_TAGS]}))
run_metadata["train_subset_sentences"] = chosen_train_size
run_metadata["validation_sentences"] = len(validation_examples)
run_metadata["test_sentences"] = len(test_examples)
run_metadata["train_subset_upos_counts"] = dict(subset_tag_counts)

Selected training sentences: 12,000 / 26,196
Validation sentences: 1,456 (all official Dev)
Test sentences: 1,455 (all official Test)
WARNING — labels still underrepresented in selected Train subset: ['X']


,UPOS,train_subset_tokens
0,ADJ,14386
1,ADP,29363
2,ADV,3398
3,AUX,7604
4,CCONJ,9154
5,DET,4333
6,INTJ,316
7,NOUN,69712
8,NUM,2648
9,PART,191


## 8. Load tokenizer and align word-level labels to subwords

A word can become multiple subword tokens. We assign the UPOS label to the first subword only and use `-100` for special tokens and continuation subwords. This is the standard Hugging Face token-classification alignment pattern.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
assert tokenizer.is_fast, "A fast tokenizer is required for word_ids() alignment."
print("Tokenizer:", tokenizer.name_or_path)
print("Tokenizer model max length:", tokenizer.model_max_length)


def tokenize_and_align(example, max_length=MAX_LENGTH):
    encoded = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=max_length,
        add_special_tokens=True,
    )
    word_ids = encoded.word_ids()
    aligned_labels = []
    previous_word_id = None
    for word_id in word_ids:
        if word_id is None:
            aligned_labels.append(-100)
        elif word_id != previous_word_id:
            aligned_labels.append(LABEL2ID[example["tags"][word_id]])
        else:
            aligned_labels.append(-100)
        previous_word_id = word_id

    represented_words = {wid for wid in word_ids if wid is not None}
    dropped_words = max(0, len(example["tokens"]) - len(represented_words))
    item = {key: value for key, value in encoded.items()}
    item["labels"] = aligned_labels
    return item, dropped_words


class TokenizedPOSDataset(Dataset):
    def __init__(self, examples):
        self.items = []
        self.dropped_words = 0
        self.sentences_with_truncation = 0
        for example in examples:
            item, dropped = tokenize_and_align(example)
            self.items.append(item)
            self.dropped_words += dropped
            if dropped > 0:
                self.sentences_with_truncation += 1

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        return self.items[index]


train_dataset = TokenizedPOSDataset(train_examples)
validation_dataset = TokenizedPOSDataset(validation_examples)
test_dataset = TokenizedPOSDataset(test_examples)

for name, dataset, examples in [
    ("train", train_dataset, train_examples),
    ("validation", validation_dataset, validation_examples),
    ("test", test_dataset, test_examples),
]:
    total_words = sum(len(ex["tokens"]) for ex in examples)
    pct = 100 * dataset.dropped_words / max(total_words, 1)
    print(f"{name:10s}: {len(dataset):,} sentences | {dataset.dropped_words:,} truncated words ({pct:.3f}%) | {dataset.sentences_with_truncation} affected sentences")

run_metadata["max_length_truncation"] = {
    name: {
        "dropped_words": ds.dropped_words,
        "sentences_with_truncation": ds.sentences_with_truncation,
    }
    for name, ds in [
        ("train", train_dataset),
        ("validation", validation_dataset),
        ("test", test_dataset),
    ]
}

if test_dataset.dropped_words / max(sum(len(x["tokens"]) for x in test_examples), 1) > 0.01:
    print("WARNING: More than 1% of Test words are truncated. Consider increasing MAX_LENGTH (up to 512) and rerun all conditions.")

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

.\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in <LOCAL_USER_HOME>\.cache\huggingface\hub\models--distilbert--distilbert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Tokenizer: distilbert/distilbert-base-multilingual-cased
Tokenizer model max length: 512
train     : 12,000 sentences | 0 truncated words (0.000%) | 0 affected sentences
validation: 1,456 sentences | 35 truncated words (0.139%) | 1 affected sentences
test      : 1,455 sentences | 0 truncated words (0.000%) | 0 affected sentences


## 9. Dynamic padding and data loaders

In [8]:
def collate_token_classification(features):
    labels = [feature["labels"] for feature in features]
    model_features = [{k: v for k, v in feature.items() if k != "labels"} for feature in features]
    batch = tokenizer.pad(model_features, padding=True, return_tensors="pt")
    max_len = batch["input_ids"].shape[1]
    padded_labels = [label + [-100] * (max_len - len(label)) for label in labels]
    batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
    return batch


def make_train_loader():
    generator = torch.Generator()
    generator.manual_seed(SEED)
    return DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        collate_fn=collate_token_classification,
        num_workers=0,
    )


def make_eval_loader(dataset):
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_token_classification,
        num_workers=0,
    )


# Shape / label sanity check.
example_batch = next(iter(make_train_loader()))
print({k: tuple(v.shape) for k, v in example_batch.items()})
assert example_batch["input_ids"].shape == example_batch["labels"].shape

{'input_ids': (8, 38), 'token_type_ids': (8, 38), 'attention_mask': (8, 38), 'labels': (8, 38)}


## 10. Evaluation functions

Accuracy is the primary metric required by the challenge. We additionally report macro-F1 and per-class precision / recall / F1 because overall accuracy can conceal poor performance on rare POS classes. All metrics are computed only on positions whose gold label is not `-100`.

In [9]:
def evaluate_model(model, dataset):
    loader = make_eval_loader(dataset)
    model.eval()
    all_true, all_pred = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = batch["labels"]
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
            )
            predictions = outputs.logits.argmax(dim=-1)
            mask = labels.ne(-100)
            all_true.extend(labels[mask].detach().cpu().tolist())
            all_pred.extend(predictions[mask].detach().cpu().tolist())

    observed_labels = sorted(set(all_true))
    metrics = {
        "token_accuracy": float(accuracy_score(all_true, all_pred)),
        "macro_f1_observed_labels": float(f1_score(
            all_true, all_pred, labels=observed_labels, average="macro", zero_division=0
        )),
        "evaluated_words": len(all_true),
    }
    report = classification_report(
        all_true,
        all_pred,
        labels=list(range(len(UPOS_TAGS))),
        target_names=UPOS_TAGS,
        output_dict=True,
        zero_division=0,
    )
    class_report = pd.DataFrame(report).T
    return metrics, class_report, all_true, all_pred

## 11. Build identical model initializations and apply freezing

The model head is randomly initialized, unlike the pretrained encoder. We reset the same seed before creating each model and verify that the initial classification-head weights match. That helps isolate freezing as the experimental variable.

In [10]:
def build_model():
    set_global_seed(SEED)
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(UPOS_TAGS),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )
    return model


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def freeze_lower_transformer_layers(model, number_to_freeze=NUM_FROZEN_LAYERS):
    layers = model.distilbert.transformer.layer
    if not (0 <= number_to_freeze <= len(layers)):
        raise ValueError(f"Requested {number_to_freeze} frozen layers, but model has {len(layers)}.")
    for layer in layers[:number_to_freeze]:
        for parameter in layer.parameters():
            parameter.requires_grad = False
    # Embeddings, upper Transformer blocks, and classifier intentionally remain trainable.
    return len(layers)


# Instantiate and immediately release a baseline model to capture its initial head weights.
baseline_model = build_model()
initial_head_weight = baseline_model.classifier.weight.detach().cpu().clone()
initial_head_bias = baseline_model.classifier.bias.detach().cpu().clone()
model_revision = getattr(baseline_model.config, "_commit_hash", None)
run_metadata["model_revision"] = model_revision
num_layers = len(baseline_model.distilbert.transformer.layer)
total_params, baseline_trainable = count_parameters(baseline_model)
print("Transformer blocks:", num_layers)
print(f"Baseline trainable parameters: {baseline_trainable:,} / {total_params:,}")

.\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in <LOCAL_USER_HOME>\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Transformer blocks: 6
Baseline trainable parameters: 134,746,384 / 134,746,384


## 12. Training loop

Both conditions use the same training subset, batch order, optimizer settings, number of epochs, and validation-based checkpoint selection. The checkpoint with the best Dev token accuracy is saved. Test results are computed only after training is complete.

In [11]:
def train_condition(model, condition_name: str):
    set_global_seed(SEED)
    model.to(DEVICE)
    train_loader = make_train_loader()
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    total_steps = EPOCHS * len(train_loader)
    warmup_steps = int(WARMUP_RATIO * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    checkpoint_path = RESULTS_DIR / f"{condition_name}_best.pt"
    best_dev_accuracy = -1.0
    history = []
    start_time = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        batches_seen = 0

        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            outputs = model(**batch)
            loss = outputs.loss
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss in {condition_name}, epoch {epoch}: {loss.item()}")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], max_norm=1.0
            )
            optimizer.step()
            scheduler.step()
            running_loss += float(loss.detach().cpu())
            batches_seen += 1

        dev_metrics, _, _, _ = evaluate_model(model, validation_dataset)
        epoch_record = {
            "condition": condition_name,
            "epoch": epoch,
            "train_loss": running_loss / max(batches_seen, 1),
            "dev_token_accuracy": dev_metrics["token_accuracy"],
            "dev_macro_f1_observed_labels": dev_metrics["macro_f1_observed_labels"],
        }
        history.append(epoch_record)
        print(epoch_record)

        if dev_metrics["token_accuracy"] > best_dev_accuracy:
            best_dev_accuracy = dev_metrics["token_accuracy"]
            torch.save(model.state_dict(), checkpoint_path)

    elapsed_seconds = time.perf_counter() - start_time
    pd.DataFrame(history).to_csv(RESULTS_DIR / f"{condition_name}_training_history.csv", index=False)

    # Reload the selected checkpoint before final Test evaluation.
    state = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(state)
    model.to(DEVICE)
    test_metrics, class_report, y_true, y_pred = evaluate_model(model, test_dataset)
    class_report.to_csv(RESULTS_DIR / f"{condition_name}_test_class_report.csv")

    output = {
        "condition": condition_name,
        "best_dev_token_accuracy": best_dev_accuracy,
        "test": test_metrics,
        "training_seconds": elapsed_seconds,
        "checkpoint": str(checkpoint_path),
    }
    with (RESULTS_DIR / f"{condition_name}_metrics.json").open("w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    return output, history, model

## 13. Run the fully fine-tuned baseline

This is the first full training run. Save its output and do not change the settings before the partial-freezing run.

In [12]:
# The model was initialized above with the fixed seed.
baseline_result, baseline_history, baseline_model = train_condition(baseline_model, "baseline_full_finetuning")
print(json.dumps(baseline_result, indent=2))

{'condition': 'baseline_full_finetuning', 'epoch': 1, 'train_loss': 0.43707298855731885, 'dev_token_accuracy': 0.9544440904746735, 'dev_macro_f1_observed_labels': 0.9381349571894642}
{'condition': 'baseline_full_finetuning', 'epoch': 2, 'train_loss': 0.11736181106418371, 'dev_token_accuracy': 0.9606164383561644, 'dev_macro_f1_observed_labels': 0.9458008900004676}
{'condition': 'baseline_full_finetuning', 'epoch': 3, 'train_loss': 0.0812515725220243, 'dev_token_accuracy': 0.9617314431347563, 'dev_macro_f1_observed_labels': 0.9449249627154327}
{
  "condition": "baseline_full_finetuning",
  "best_dev_token_accuracy": 0.9617314431347563,
  "test": {
    "token_accuracy": 0.9636182820204698,
    "macro_f1_observed_labels": 0.887309046743356,
    "evaluated_words": 24133
  },
  "training_seconds": 779.4978558998555,
  "checkpoint": "results/baseline_full_finetuning_best.pt"
}


## 14. Run partial freezing (lower 3 Transformer blocks)

Recreate the model from the same pretrained checkpoint and same seed. Verify that the initial task-head weights match the baseline's initial head weights before freezing.

In [17]:
from collections import Counter
import pandas as pd

audit_rows = []

for split, examples in raw_splits.items():
    counts = Counter(
        tag
        for example in examples
        for tag in example["tags"]
    )

    for tag in UPOS_TAGS:
        audit_rows.append({
            "split": split,
            "upos": tag,
            "count": counts[tag]
        })

upos_audit = pd.DataFrame(audit_rows)

display(
    upos_audit.pivot(
        index="upos",
        columns="split",
        values="count"
    ).fillna(0).astype(int)
)

upos_audit.to_csv(
    RESULTS_DIR / "upos_split_counts.csv",
    index=False
)

print("Dataset audit saved.")

split,test,train,validation
upos,,,
ADJ,1652,31811,1887
ADP,3407,64113,3567
ADV,384,7460,419
AUX,899,16684,913
CCONJ,1026,20149,1114
DET,491,9409,538
INTJ,27,629,41
NOUN,8219,151678,8317
NUM,293,5594,294


Dataset audit saved.


In [13]:
del baseline_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

partial_model = build_model()
assert torch.equal(partial_model.classifier.weight.detach().cpu(), initial_head_weight), \
    "Initial classifier weights differ between baseline and partial model. Check seed / software versions."
assert torch.equal(partial_model.classifier.bias.detach().cpu(), initial_head_bias), \
    "Initial classifier bias differs between baseline and partial model. Check seed / software versions."

actual_num_layers = freeze_lower_transformer_layers(partial_model, NUM_FROZEN_LAYERS)
total_params, partial_trainable = count_parameters(partial_model)
print(f"Frozen lower Transformer blocks: {NUM_FROZEN_LAYERS} / {actual_num_layers}")
print(f"Partial model trainable parameters: {partial_trainable:,} / {total_params:,}")
print(f"Trainable parameter fraction: {100 * partial_trainable / total_params:.2f}%")

partial_result, partial_history, partial_model = train_condition(partial_model, "partial_freezing_lower3")
print(json.dumps(partial_result, indent=2))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Frozen lower Transformer blocks: 3 / 6
Partial model trainable parameters: 113,482,768 / 134,746,384
Trainable parameter fraction: 84.22%
{'condition': 'partial_freezing_lower3', 'epoch': 1, 'train_loss': 0.500737805125614, 'dev_token_accuracy': 0.9462408410321759, 'dev_macro_f1_observed_labels': 0.9214444997223378}
{'condition': 'partial_freezing_lower3', 'epoch': 2, 'train_loss': 0.1489973453618586, 'dev_token_accuracy': 0.9529706913029627, 'dev_macro_f1_observed_labels': 0.9332351625781009}
{'condition': 'partial_freezing_lower3', 'epoch': 3, 'train_loss': 0.11918243449678023, 'dev_token_accuracy': 0.9548423064670277, 'dev_macro_f1_observed_labels': 0.9369645741978382}
{
  "condition": "partial_freezing_lower3",
  "best_dev_token_accuracy": 0.9548423064670277,
  "test": {
    "token_accuracy": 0.9577756598848051,
    "macro_f1_observed_labels": 0.8807321293005124,
    "evaluated_words": 24133
  },
  "training_seconds": 674.8067812998779,
  "checkpoint": "results/partial_freezing_low

## 15. Compare the two conditions

In [14]:
comparison = pd.DataFrame([
    {
        "condition": "Baseline: full fine-tuning",
        "trainable_parameters": baseline_trainable,
        "best_dev_accuracy": baseline_result["best_dev_token_accuracy"],
        "test_accuracy": baseline_result["test"]["token_accuracy"],
        "test_macro_f1": baseline_result["test"]["macro_f1_observed_labels"],
        "training_seconds": baseline_result["training_seconds"],
    },
    {
        "condition": "Partial: freeze lower 3 blocks",
        "trainable_parameters": partial_trainable,
        "best_dev_accuracy": partial_result["best_dev_token_accuracy"],
        "test_accuracy": partial_result["test"]["token_accuracy"],
        "test_macro_f1": partial_result["test"]["macro_f1_observed_labels"],
        "training_seconds": partial_result["training_seconds"],
    },
])
comparison["accuracy_difference_vs_baseline"] = comparison["test_accuracy"] - comparison.loc[0, "test_accuracy"]
comparison["training_time_change_vs_baseline_pct"] = (
    100 * (comparison["training_seconds"] - comparison.loc[0, "training_seconds"])
    / comparison.loc[0, "training_seconds"]
)
display(comparison)
comparison.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)

print("Accuracy difference (partial - baseline):",
      partial_result["test"]["token_accuracy"] - baseline_result["test"]["token_accuracy"])
print("Training-time difference in seconds:",
      partial_result["training_seconds"] - baseline_result["training_seconds"])

,condition,trainable_parameters,best_dev_accuracy,test_accuracy,test_macro_f1,training_seconds,accuracy_difference_vs_baseline,training_time_change_vs_baseline_pct
0,Baseline: full fine-tuning,134746384,0.961731,0.963618,0.887309,779.497856,0.000000,0.000000
1,Partial: freeze lower 3 blocks,113482768,0.954842,0.957776,0.880732,674.806781,-0.005843,-13.430579


Accuracy difference (partial - baseline): -0.005842622135664777
Training-time difference in seconds: -104.69107459997758


## 16. Class-level error analysis

Compare class-level F1 to see whether freezing disproportionately harms particular grammatical categories. The report includes support so that low-support classes are not over-interpreted.

In [15]:
baseline_report = pd.read_csv(RESULTS_DIR / "baseline_full_finetuning_test_class_report.csv", index_col=0)
partial_report = pd.read_csv(RESULTS_DIR / "partial_freezing_lower3_test_class_report.csv", index_col=0)

class_rows = []
for tag in UPOS_TAGS:
    b = baseline_report.loc[tag]
    p = partial_report.loc[tag]
    class_rows.append({
        "UPOS": tag,
        "baseline_precision": b["precision"],
        "partial_precision": p["precision"],
        "baseline_recall": b["recall"],
        "partial_recall": p["recall"],
        "baseline_f1": b["f1-score"],
        "partial_f1": p["f1-score"],
        "support": int(b["support"]),
        "f1_difference_partial_minus_baseline": p["f1-score"] - b["f1-score"],
    })
class_comparison = pd.DataFrame(class_rows).sort_values("f1_difference_partial_minus_baseline")
display(class_comparison)
class_comparison.to_csv(RESULTS_DIR / "class_level_comparison.csv", index=False)

,UPOS,baseline_precision,partial_precision,baseline_recall,partial_recall,baseline_f1,partial_f1,support,f1_difference_partial_minus_baseline
6,INTJ,0.840000,0.769231,0.777778,0.740741,0.807692,0.754717,27,-0.052975
0,ADJ,0.890741,0.870581,0.873487,0.834746,0.882029,0.852287,1652,-0.029743
11,PROPN,0.815981,0.806189,0.909991,0.891089,0.860426,0.846516,1111,-0.013910
7,NOUN,0.962648,0.951658,0.950116,0.946100,0.956341,0.948871,8219,-0.007469
2,ADV,0.934037,0.938005,0.921875,0.906250,0.927916,0.921854,384,-0.006062
8,NUM,0.941368,0.938111,0.986348,0.982935,0.963333,0.960000,293,-0.003333
14,VERB,0.990015,0.987421,0.992953,0.989985,0.991481,0.988702,2696,-0.002780
5,DET,0.955645,0.953629,0.965377,0.963340,0.960486,0.958460,491,-0.002026
13,SCONJ,0.987241,0.984102,0.979430,0.979430,0.983320,0.981761,632,-0.001560
3,AUX,0.988877,0.987764,0.988877,0.987764,0.988877,0.987764,899,-0.001112


## 17. Export metadata and final run summary

In [16]:
run_metadata["baseline_result"] = baseline_result
run_metadata["partial_freezing_result"] = partial_result
run_metadata["baseline_trainable_parameters"] = baseline_trainable
run_metadata["partial_trainable_parameters"] = partial_trainable
run_metadata["total_parameters"] = total_params
run_metadata["frozen_transformer_blocks"] = NUM_FROZEN_LAYERS
run_metadata["upos_tags"] = UPOS_TAGS
run_metadata["wall_clock_note"] = "Training seconds are measured per condition inside train_condition; download/tokenization time is excluded."

with (RESULTS_DIR / "run_metadata.json").open("w", encoding="utf-8") as f:
    json.dump(run_metadata, f, ensure_ascii=False, indent=2)

print("Saved result files:")
for path in sorted(RESULTS_DIR.iterdir()):
    print(f" - {path.name} ({path.stat().st_size:,} bytes)")

Saved result files:
 - baseline_full_finetuning_best.pt (539,029,243 bytes)
 - baseline_full_finetuning_metrics.json (414 bytes)
 - baseline_full_finetuning_test_class_report.csv (1,228 bytes)
 - baseline_full_finetuning_training_history.csv (333 bytes)
 - class_level_comparison.csv (2,146 bytes)
 - model_comparison.csv (443 bytes)
 - partial_freezing_lower3_best.pt (539,029,135 bytes)
 - partial_freezing_lower3_metrics.json (413 bytes)
 - partial_freezing_lower3_test_class_report.csv (1,231 bytes)
 - partial_freezing_lower3_training_history.csv (328 bytes)
 - run_metadata.json (3,168 bytes)


## 18. Discussion prompts — complete after inspecting your actual results

Do not write generic explanations. Answer these questions using the generated tables and your own observations.

1. Did partial freezing increase or decrease test token accuracy? By how many percentage points?
2. Did macro-F1 move in the same direction as accuracy? What does that suggest about class imbalance?
3. Which UPOS labels gained or lost the most F1? Check their support before interpreting the change.
4. Did training time decrease? Note that fewer trainable parameters do **not** guarantee a proportional wall-clock speedup because the frozen layers still perform forward computation.
5. How many parameters were trainable in each condition? What fraction of the full model was updated?
6. Were any words truncated at MAX_LENGTH=256? If yes, decide whether to increase the limit and rerun **both** conditions.
7. What limitations remain? Consider one language, one treebank, one freezing pattern, one subset size, three epochs, random variation, and domain coverage.
8. What is the next controlled experiment? A reasonable next ablation is to compare freezing 1, 2, 3, or 4 lower Transformer blocks while keeping every other setting fixed. Do not add these runs until the primary comparison is complete and documented.

### Suggested future direction (hypothesis, not a result)
A follow-up could test whether the best freezing depth depends on the target language's morphological properties or on the similarity between the target language and the model's pretraining languages. This should be motivated with relevant literature and tested across more than one language before making broad claims.

---

### Answers to the Discussion Questions

**1. Effect on test token accuracy**

Partial freezing decreased test token accuracy from 96.36% to 95.78%, a decrease of approximately 0.584 percentage points.

**2. Macro-F1 comparison**

Macro-F1 also decreased, from 88.73% to 88.07%, a decrease of approximately 0.658 percentage points. Both metrics moved in the same direction. The difference between accuracy and macro-F1 indicates that performance should also be examined at the individual UPOS-label level, particularly for less frequent classes.

**3. Class-level F1 changes**

INTJ experienced the largest F1 decrease (-0.0530), with 27 test examples. ADJ decreased by 0.0297 (1,652 examples), while PROPN decreased by 0.0139 (1,111 examples). PART gained the most (+0.0174), but it had only 28 test examples. These low-support results should be interpreted cautiously.

**4. Training-time comparison**

Training time decreased from 779.50 seconds to 674.81 seconds. This is a reduction of approximately 104.69 seconds, or 13.43%. Frozen Transformer blocks still perform forward computation, so the reduction in training time is not expected to match the reduction in trainable parameters exactly.

**5. Trainable parameters**

Full fine-tuning updated 134,746,384 parameters, representing 100% of the model. Partial freezing updated 113,482,768 parameters, representing approximately 84.22% of the total. Therefore, approximately 15.78% of the parameters were excluded from gradient updates.

**6. Sequence truncation**

At MAX_LENGTH=256, no words were truncated in the training or test sets. The validation set had 35 truncated words across one sentence, representing 0.139% of its words. The current comparison will retain MAX_LENGTH=256 because the observed truncation is limited, but this should be documented as a limitation. If the maximum length is changed later, both experimental conditions must be rerun under the same setting.

**7. Remaining limitations**

The experiment covers only Persian and one treebank, UD Persian PerDT. It uses a subset of 12,000 training sentences, one freezing configuration (three lower Transformer blocks), and three training epochs. Only one random seed was used, so run-to-run variation has not been measured. Domain coverage may also limit generalization. The X label has only one test example, making its class-level evaluation unreliable.

**8. Next controlled experiment**

A suitable next experiment is to compare freezing 1, 2, 3, and 4 lower Transformer blocks. The dataset splits, random seed, optimizer, learning rate, batch size, number of epochs, and evaluation procedure should remain unchanged. Test accuracy, macro-F1, training time, and trainable parameter count should be recorded for each condition.


## 19. Required attribution and source notes

- Treebank: **UD Persian PerDT**, Universal Dependencies. The official UD page identifies Persian (`fa`), describes its genre coverage and annotation provenance, and lists the UPOS tags used by the treebank. License: **CC BY-SA 4.0**. Cite the treebank and its underlying publication in your report.
- Model: `distilbert/distilbert-base-multilingual-cased`. Its model card describes a 6-layer, 768-dimensional, 12-head multilingual DistilBERT trained on 104 languages.
- Token-label alignment: follow the Hugging Face token-classification guidance: first subword receives the word label; special tokens and continuation subwords receive `-100`.

Record the exact dataset checksums and model revision saved in `results/run_metadata.json` with your report.